# ModernBERT LoRA 3-Fold Inference Submission

This notebook creates the final ModernBERT LoRA submission by loading all three trained fold adapters.

Expected Kaggle inputs:
- ModernBERT base model: `answer-ai/modernbert/Transformers/large/2`
- Private LLM classification dataset, or the competition data
- Kernel output `sarthak11j/try-sj-12` containing `fold_1`
- Kernel output `sarthak11j/try-sj-14` containing `fold_2` and `fold_3`

The notebook runs original plus swapped response inference for each fold, averages all three fold predictions, and writes `/kaggle/working/submission.csv`.


## Offline Dependency Bootstrap

In [ ]:

from __future__ import annotations

import gc
import importlib.metadata as importlib_metadata
import json
import os
import re
import subprocess
import sys
import zipfile
from dataclasses import dataclass
from pathlib import Path

os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
os.environ.setdefault("HF_DATASETS_OFFLINE", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

MIN_DEPENDENCIES = {
    "transformers": "4.52.1",
    "accelerate": "0.29.3",
    "peft": "0.11.1",
}


def parse_version_tuple(version: str) -> tuple[int, ...]:
    parts = re.findall(r"\d+", version)
    return tuple(int(part) for part in parts[:3]) if parts else (0,)


def installed_version(package_name: str) -> str | None:
    try:
        return importlib_metadata.version(package_name)
    except importlib_metadata.PackageNotFoundError:
        return None


def package_is_usable(package_name: str, minimum_version: str) -> bool:
    version = installed_version(package_name)
    if version is None:
        return False
    return parse_version_tuple(version) >= parse_version_tuple(minimum_version)


def offline_wheel_dirs() -> list[str]:
    input_root = Path("/kaggle/input")
    if not input_root.exists():
        return []
    return sorted({str(path.parent) for path in input_root.rglob("*.whl")})


def install_requirement_offline(package_name: str, requirement: str) -> None:
    wheel_dirs = offline_wheel_dirs()
    if not wheel_dirs:
        raise RuntimeError(
            f"{package_name} is missing or too old, and no offline wheels were found under /kaggle/input. "
            "Attach a dataset containing compatible wheels."
        )

    command = [sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check", "--no-index"]
    for wheel_dir in wheel_dirs:
        command.extend(["--find-links", wheel_dir])
    command.append(requirement)
    print("Running:", " ".join(command))
    subprocess.check_call(command)


def ensure_dependencies() -> None:
    for package_name, minimum_version in MIN_DEPENDENCIES.items():
        version = installed_version(package_name)
        if package_is_usable(package_name, minimum_version):
            print(f"{package_name} {version} is available.")
            continue
        requirement = f"{package_name}>={minimum_version}"
        print(f"Installing {requirement} from offline wheels.")
        install_requirement_offline(package_name, requirement)


ensure_dependencies()


## Imports And Configuration

In [ ]:

import numpy as np
import pandas as pd
import torch
from peft import PeftModel
from torch.utils.data import DataLoader
from transformers import AutoConfig, AutoModelForSequenceClassification, AutoTokenizer

LABEL_COLUMNS = ["winner_model_a", "winner_model_b", "winner_tie"]
NUM_LABELS = len(LABEL_COLUMNS)
SWAP_LABEL_MAP = np.array([1, 0, 2], dtype=np.int64)

MAX_LENGTH = int(os.environ.get("LLM_MAX_LENGTH", "2048"))
INFER_BATCH_SIZE = int(os.environ.get("LLM_INFER_BATCH_SIZE", "2"))
NUM_WORKERS = int(os.environ.get("LLM_NUM_WORKERS", "2"))
USE_FP16 = os.environ.get("LLM_USE_FP16", "1").strip().lower() in {"1", "true", "yes", "on"}
CLASSIFIER_POOLING = os.environ.get("LLM_CLASSIFIER_POOLING", "mean").strip().lower()
PROBABILITY_EPS = float(os.environ.get("LLM_PROBABILITY_EPS", "1e-7"))

PROMPT_SHARE = 0.18
RESPONSE_A_SHARE = 0.41
RESPONSE_B_SHARE = 0.41
PROMPT_HEAD_RATIO = 0.80
RESPONSE_HEAD_RATIO = 0.72

OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()


## Locate Data, Base Model, And All Fold Adapters


In [ ]:
BASE_MODEL_PATHS = [
    "/kaggle/input/modernbert/transformers/large/2",
    "/kaggle/input/modernbert/Transformers/large/2",
    "/kaggle/input/models/answer-ai/modernbert/transformers/large/2",
    "/kaggle/input/models/answer-ai/modernbert/Transformers/large/2",
    "/kaggle/input/models/answer-ai/modernbert/pytorch/large/2",
    "/kaggle/input/answer-ai-modernbert/transformers/large/2",
    "/kaggle/input/answer-ai-modernbert/Transformers/large/2",
    "/kaggle/input/modernbert-large",
    "/kaggle/input/answerdotai-modernbert-large",
    "/kaggle/input/modernbert-large-offline",
]

PRIVATE_DATASET_INPUT_DIRS = [
    Path("/kaggle/input/llm-classification-finetuning-private-data"),
    Path("/kaggle/input/llm-classification-finetuning-private"),
    Path("/kaggle/input/llm-classification-private-data"),
]
COMPETITION_INPUT_DIRS = [
    Path("/kaggle/input/llm-classification-finetuning"),
    Path("/kaggle/input/competitions/llm-classification-finetuning"),
]
PRIVATE_DATASET_HINTS = ("llm-classification", "finetuning", "private")
EXCLUDED_DATASET_HINTS = ("nvidia", "nemotron", "reasoning-challenge")
REQUIRED_FOLDS = [1, 2, 3]


def find_data_dir() -> Path:
    env_data_dir = os.environ.get("LLM_DATA_DIR")
    if env_data_dir:
        path = Path(env_data_dir)
        if (path / "test.csv").exists():
            print(f"Using data directory from LLM_DATA_DIR: {path}")
            return path
        raise FileNotFoundError(f"LLM_DATA_DIR does not contain test.csv: {path}")

    for path in PRIVATE_DATASET_INPUT_DIRS:
        if (path / "test.csv").exists():
            print(f"Using private LLM dataset: {path}")
            return path

    for path in COMPETITION_INPUT_DIRS:
        if (path / "test.csv").exists():
            print(f"Using competition data: {path}")
            return path

    candidates = []
    for test_path in sorted(Path("/kaggle/input").rglob("test.csv")):
        parent = test_path.parent
        lowered = str(parent).lower()
        if any(hint in lowered for hint in EXCLUDED_DATASET_HINTS):
            continue
        score = sum(hint in lowered for hint in PRIVATE_DATASET_HINTS)
        candidates.append((score, str(parent), parent))
    if candidates:
        candidates.sort(key=lambda item: (-item[0], item[1]))
        selected = candidates[0][2]
        print(f"Discovered LLM data directory: {selected}")
        return selected
    raise FileNotFoundError("Could not find LLM Classification test.csv under /kaggle/input.")


def find_base_model_dir() -> Path:
    env_path = os.environ.get("LLM_MODEL_PATH")
    if env_path and Path(env_path).exists():
        return Path(env_path)
    for path in BASE_MODEL_PATHS:
        if Path(path).exists():
            return Path(path)
    for config_path in sorted(Path("/kaggle/input").rglob("config.json")):
        try:
            payload = json.loads(config_path.read_text(encoding="utf-8"))
        except Exception:
            continue
        model_type = str(payload.get("model_type", "")).lower()
        architectures = " ".join(str(item) for item in payload.get("architectures", [])).lower()
        if "modernbert" in model_type or "modernbert" in architectures:
            return config_path.parent
    raise FileNotFoundError("Could not find attached offline ModernBERT base model.")


def extract_adapter_zips() -> Path:
    extract_root = OUTPUT_DIR / "extracted_modernbert_adapters"
    for zip_path in Path("/kaggle/input").rglob("*.zip"):
        lower_name = zip_path.name.lower()
        if "fold" not in lower_name and "adapter" not in lower_name and "lora" not in lower_name:
            continue
        target = extract_root / zip_path.stem
        if target.exists():
            continue
        print(f"Extracting possible adapter archive: {zip_path} -> {target}")
        target.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(target)
    return extract_root


def infer_fold_number(path: Path) -> int | None:
    text = str(path).lower().replace("-", "_")
    for fold in REQUIRED_FOLDS:
        if f"fold_{fold}" in text or f"fold{fold}" in text:
            return fold
    return None


def adapter_score(path: Path, fold: int) -> tuple[int, str]:
    text = str(path).lower().replace("-", "_")
    score = 0
    if f"fold_{fold}" in text:
        score -= 20
    if fold == 1 and "try_sj_12" in text:
        score -= 10
    if fold in {2, 3} and "try_sj_14" in text:
        score -= 10
    if "working" in text:
        score += 5
    return (score, str(path))


def parse_explicit_adapter_dirs() -> dict[int, Path]:
    raw = os.environ.get("LLM_ADAPTER_DIRS", "").strip()
    if not raw:
        return {}
    selected = {}
    for item in raw.split(","):
        path = Path(item.strip())
        if not path:
            continue
        if not (path / "adapter_config.json").exists():
            raise FileNotFoundError(f"Adapter dir is missing adapter_config.json: {path}")
        fold = infer_fold_number(path)
        if fold is None:
            raise ValueError(f"Could not infer fold number from adapter path: {path}")
        selected[fold] = path
    return selected


def find_adapter_dirs() -> dict[int, Path]:
    explicit = parse_explicit_adapter_dirs()
    if explicit:
        missing = [fold for fold in REQUIRED_FOLDS if fold not in explicit]
        if missing:
            raise ValueError(f"LLM_ADAPTER_DIRS is missing folds: {missing}")
        return explicit

    extract_root = extract_adapter_zips()
    roots = [Path("/kaggle/input"), extract_root, Path("/kaggle/working")]
    candidates_by_fold = {fold: [] for fold in REQUIRED_FOLDS}
    for root in roots:
        if not root.exists():
            continue
        for config_path in root.rglob("adapter_config.json"):
            parent = config_path.parent
            has_weights = (parent / "adapter_model.safetensors").exists() or (parent / "adapter_model.bin").exists()
            if not has_weights:
                continue
            fold = infer_fold_number(parent)
            if fold in candidates_by_fold:
                candidates_by_fold[fold].append(parent)

    selected = {}
    for fold, candidates in candidates_by_fold.items():
        unique_candidates = sorted(set(candidates), key=lambda path: adapter_score(path, fold))
        print(f"Fold {fold} adapter candidates:")
        for candidate in unique_candidates[:10]:
            print("-", candidate)
        if not unique_candidates:
            raise FileNotFoundError(
                f"Could not find PEFT adapter for fold_{fold}. Attach try-sj-12 and try-sj-14 outputs, "
                "or set LLM_ADAPTER_DIRS to comma-separated fold adapter paths."
            )
        selected[fold] = unique_candidates[0]
    return selected


data_dir = find_data_dir()
base_model_dir = find_base_model_dir()
adapter_dirs = find_adapter_dirs()

print("Data dir:       ", data_dir)
print("Base model dir: ", base_model_dir)
print("Selected adapters:")
for fold in REQUIRED_FOLDS:
    print(f"- fold_{fold}: {adapter_dirs[fold]}")


## Tokenization Helpers

In [ ]:

@dataclass(frozen=True)
class EncodedRow:
    prompt_ids: list[int]
    response_a_ids: list[int]
    response_b_ids: list[int]


def truncate_head_tail(token_ids: list[int], budget: int, head_ratio: float) -> list[int]:
    if budget <= 0:
        return []
    if len(token_ids) <= budget:
        return token_ids
    head_count = max(1, int(round(budget * head_ratio)))
    head_count = min(head_count, budget - 1)
    tail_count = budget - head_count
    if tail_count <= 0:
        return token_ids[:budget]
    return token_ids[:head_count] + token_ids[-tail_count:]


def allocate_budgets(lengths: list[int], total_budget: int, shares: list[float]) -> list[int]:
    budgets = [min(length, int(total_budget * share)) for length, share in zip(lengths, shares)]
    remaining = max(0, total_budget - sum(budgets))
    while remaining > 0:
        candidate = max(range(len(lengths)), key=lambda idx: (lengths[idx] - budgets[idx], lengths[idx]))
        if budgets[candidate] >= lengths[candidate]:
            break
        budgets[candidate] += 1
        remaining -= 1
    return budgets


def build_model_inputs(encoded_row: EncodedRow, tokenizer, max_length: int, swap_responses: bool) -> tuple[list[int], list[int]]:
    prompt_ids = encoded_row.prompt_ids
    response_a_ids = encoded_row.response_b_ids if swap_responses else encoded_row.response_a_ids
    response_b_ids = encoded_row.response_a_ids if swap_responses else encoded_row.response_b_ids

    content_budget = max_length - 4
    budgets = allocate_budgets(
        lengths=[len(prompt_ids), len(response_a_ids), len(response_b_ids)],
        total_budget=content_budget,
        shares=[PROMPT_SHARE, RESPONSE_A_SHARE, RESPONSE_B_SHARE],
    )

    prompt_final = truncate_head_tail(prompt_ids, budgets[0], PROMPT_HEAD_RATIO)
    response_a_final = truncate_head_tail(response_a_ids, budgets[1], RESPONSE_HEAD_RATIO)
    response_b_final = truncate_head_tail(response_b_ids, budgets[2], RESPONSE_HEAD_RATIO)

    input_ids = [
        tokenizer.cls_token_id,
        *prompt_final,
        tokenizer.sep_token_id,
        *response_a_final,
        tokenizer.sep_token_id,
        *response_b_final,
        tokenizer.sep_token_id,
    ]
    return input_ids, [1] * len(input_ids)


def encode_texts(tokenizer, texts: list[str], batch_size: int = 256) -> list[list[int]]:
    all_ids = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start : start + batch_size]
        tokenized = tokenizer(batch, add_special_tokens=False, truncation=False, verbose=False)
        all_ids.extend(tokenized["input_ids"])
    return all_ids


def pretokenize_dataframe(df: pd.DataFrame, tokenizer) -> list[EncodedRow]:
    prompt_ids = encode_texts(tokenizer, ("Prompt:\n" + df["prompt"].fillna("").astype(str)).tolist())
    response_a_ids = encode_texts(tokenizer, ("Response A:\n" + df["response_a"].fillna("").astype(str)).tolist())
    response_b_ids = encode_texts(tokenizer, ("Response B:\n" + df["response_b"].fillna("").astype(str)).tolist())
    return [EncodedRow(p, a, b) for p, a, b in zip(prompt_ids, response_a_ids, response_b_ids)]


## Dataset And Prediction Utilities

In [ ]:

class PreferenceDataset:
    def __init__(self, encoded_rows, row_indices, tokenizer, max_length, swap_flags=None):
        self.encoded_rows = encoded_rows
        self.row_indices = row_indices.astype(np.int64)
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.swap_flags = swap_flags.astype(bool) if swap_flags is not None else np.zeros(len(self.row_indices), dtype=bool)

    def __len__(self):
        return len(self.row_indices)

    def __getitem__(self, index):
        row_index = int(self.row_indices[index])
        swap = bool(self.swap_flags[index])
        input_ids, attention_mask = build_model_inputs(self.encoded_rows[row_index], self.tokenizer, self.max_length, swap)
        return {"input_ids": input_ids, "attention_mask": attention_mask}


def build_collate_fn(tokenizer):
    pad_id = tokenizer.pad_token_id

    def collate_fn(batch):
        max_len = max(len(item["input_ids"]) for item in batch)
        input_ids, attention_mask = [], []
        for item in batch:
            pad_width = max_len - len(item["input_ids"])
            input_ids.append(item["input_ids"] + [pad_id] * pad_width)
            attention_mask.append(item["attention_mask"] + [0] * pad_width)
        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
        }

    return collate_fn


def normalize_probabilities(probs: np.ndarray) -> np.ndarray:
    probs = np.asarray(probs, dtype=np.float64)
    probs = np.clip(probs, PROBABILITY_EPS, 1.0 - PROBABILITY_EPS)
    return (probs / probs.sum(axis=1, keepdims=True)).astype(np.float32)


@np.errstate(over="ignore")
def softmax_np(logits: np.ndarray) -> np.ndarray:
    shifted = logits - logits.max(axis=1, keepdims=True)
    exp_scores = np.exp(shifted)
    return normalize_probabilities(exp_scores / exp_scores.sum(axis=1, keepdims=True))


def autocast_context(device):
    if hasattr(torch, "amp") and hasattr(torch.amp, "autocast"):
        return torch.amp.autocast("cuda", enabled=USE_FP16 and device.type == "cuda")
    return torch.cuda.amp.autocast(enabled=USE_FP16 and device.type == "cuda")


def predict_probabilities(model, loader, device) -> np.ndarray:
    model.eval()
    all_logits = []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device, non_blocking=True)
            attention_mask = batch["attention_mask"].to(device, non_blocking=True)
            with autocast_context(device):
                logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            all_logits.append(logits.detach().cpu().to(torch.float32).numpy())
    return softmax_np(np.vstack(all_logits))


def predict_with_swap_tta(model, loader_original, loader_swapped, device) -> np.ndarray:
    original_probs = predict_probabilities(model, loader_original, device)
    swapped_probs = predict_probabilities(model, loader_swapped, device)
    swapped_back = swapped_probs[:, SWAP_LABEL_MAP]
    return normalize_probabilities(0.5 * (original_probs + swapped_back))


## Load Three Fold Adapters And Create Submission


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

config = AutoConfig.from_pretrained(base_model_dir, num_labels=NUM_LABELS)
if hasattr(config, "classifier_pooling"):
    config.classifier_pooling = CLASSIFIER_POOLING
if hasattr(config, "reference_compile"):
    config.reference_compile = False

model_max_length = getattr(config, "max_position_embeddings", None)
effective_max_length = min(MAX_LENGTH, int(model_max_length)) if model_max_length else MAX_LENGTH
print(f"Max length requested/effective: {MAX_LENGTH}/{effective_max_length}")

first_adapter_dir = adapter_dirs[REQUIRED_FOLDS[0]]
tokenizer_source = first_adapter_dir if (first_adapter_dir / "tokenizer.json").exists() else base_model_dir
tokenizer = AutoTokenizer.from_pretrained(tokenizer_source)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token or tokenizer.sep_token

print("Reading test data...")
test_df = pd.read_csv(data_dir / "test.csv")
required = ["id", "prompt", "response_a", "response_b"]
missing = [col for col in required if col not in test_df.columns]
if missing:
    raise ValueError(f"test.csv is missing required columns: {missing}")

print(f"Test rows: {len(test_df):,}")
print("Pretokenizing test fields...")
encoded_test = pretokenize_dataframe(test_df, tokenizer)
indices = np.arange(len(test_df), dtype=np.int64)
collate_fn = build_collate_fn(tokenizer)

loader_original = DataLoader(
    PreferenceDataset(encoded_test, indices, tokenizer, effective_max_length),
    batch_size=INFER_BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
)
loader_swapped = DataLoader(
    PreferenceDataset(encoded_test, indices, tokenizer, effective_max_length, swap_flags=np.ones(len(indices), dtype=bool)),
    batch_size=INFER_BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
)

fold_probs = []
for fold in REQUIRED_FOLDS:
    adapter_dir = adapter_dirs[fold]
    print(f"Loading base ModernBERT for fold_{fold}...")
    base_model = AutoModelForSequenceClassification.from_pretrained(base_model_dir, config=config)
    print(f"Loading PEFT adapter fold_{fold}: {adapter_dir}")
    model = PeftModel.from_pretrained(base_model, adapter_dir)
    model.to(device)
    model.eval()

    print(f"Predicting with fold_{fold} adapter and swap TTA...")
    probs = predict_with_swap_tta(model, loader_original, loader_swapped, device)
    fold_probs.append(probs)

    del model, base_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

probs = normalize_probabilities(np.mean(np.stack(fold_probs, axis=0), axis=0))
submission_df = pd.DataFrame({
    "id": test_df["id"],
    LABEL_COLUMNS[0]: probs[:, 0],
    LABEL_COLUMNS[1]: probs[:, 1],
    LABEL_COLUMNS[2]: probs[:, 2],
})

row_sum_error = float(np.abs(submission_df[LABEL_COLUMNS].sum(axis=1).values - 1.0).max())
if row_sum_error > 1e-5:
    raise ValueError(f"Submission probability rows do not sum to 1. Max error: {row_sum_error}")

out_path = OUTPUT_DIR / "submission.csv"
submission_df.to_csv(out_path, index=False)
metrics = {
    "folds": REQUIRED_FOLDS,
    "adapter_dirs": {f"fold_{fold}": str(adapter_dirs[fold]) for fold in REQUIRED_FOLDS},
    "base_model_dir": str(base_model_dir),
    "data_dir": str(data_dir),
    "max_length_requested": MAX_LENGTH,
    "max_length_effective": effective_max_length,
    "test_rows": int(len(test_df)),
}
(OUTPUT_DIR / "modernbert_3fold_inference_metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
print(f"Saved submission to {out_path}")
print(submission_df.head())


## Expected Output

The notebook writes `/kaggle/working/submission.csv` and `/kaggle/working/modernbert_3fold_inference_metrics.json`. Submit the notebook output to the LLM Classification Finetuning competition.
